# CIE6032：Qwen 0.5B LoRA（免费 Colab 优先）
先选择“运行时 → 更改运行时类型 → GPU”，再逐格运行。不要一次运行全部单元格。
本 notebook 在本机做了语法与打包检查，尚未在你的 Colab 账号执行。免费 GPU 不保证分配或持续可用。
先运行 10 步短测，再运行 1000 条，最后才考虑 5000 条。不会自动购买算力。
上传包仅含公开数据的派生训练文件、代码和许可证，不包含原始课程文件、账号密钥或约 1GB 模型。

In [ ]:
from google.colab import files
from pathlib import Path
import zipfile, hashlib, json, os
uploaded = files.upload()  # 选择本地 intent_router_colab.zip
assert 'intent_router_colab.zip' in uploaded
root = Path('/content/intent_router')
root.mkdir(exist_ok=True)
with zipfile.ZipFile('intent_router_colab.zip') as z:
    for entry in z.infolist():
        target = (root / entry.filename).resolve()
        assert target.is_relative_to(root.resolve()), 'Unsafe archive path'
    z.extractall(root)
os.chdir(root)
for name, sha in json.loads(Path('bundle_manifest.json').read_text()).items():
    assert hashlib.sha256(Path(name).read_bytes()).hexdigest() == sha
print('上传包校验通过')


安装固定框架版本，保留 Colab 自带的 CUDA torch/vision/audio。若依赖解析失败，不要强行忽略依赖。安装在子进程中，后续训练也使用新进程。

In [ ]:
import subprocess, sys, importlib.metadata as metadata
constraints = []
for name in ['torch', 'torchvision', 'torchaudio']:
    try: constraints.append(name + '==' + metadata.version(name))
    except metadata.PackageNotFoundError: pass
Path('colab_constraints.txt').write_text('\n'.join(constraints))
subprocess.run([sys.executable, '-m', 'pip', 'install', '-c', 'colab_constraints.txt', 'llamafactory==0.9.4'], check=True)
print('框架安装完成')


In [ ]:
# 下载与本机相同 revision 的模型；不需要付费 API。
subprocess.run([sys.executable, 'download_model.py'], check=True)
subprocess.run([sys.executable, '-c', "import torch; assert torch.cuda.is_available(); print(torch.cuda.get_device_name()); print('BF16:',torch.cuda.is_bf16_supported())"], check=True)


挂载 Drive 保存 checkpoint，避免运行时回收后全部丢失。授权只在 Colab 页面进行。每次运行新目录，checkpoint 每 25 步保存一次，保留最近两个；最后一次未保存的进度仍可能丢失。

In [ ]:
from google.colab import drive
from datetime import datetime
drive.mount('/content/drive')
result_root = Path('/content/drive/MyDrive/CIE6032-intent-router')
result_root.mkdir(exist_ok=True)


先运行这一格：10 个 optimizer step 的 GPU 短测。完成后查看 gpu_measurement.json 的显存和时间；短测 adapter 不用于正式效果对比。

In [ ]:
smoke_out = result_root / ('smoke-' + datetime.now().strftime('%Y%m%d-%H%M%S'))
subprocess.run([sys.executable, 'colab_train.py', '--smoke', '--output', str(smoke_out)], check=True)
print((smoke_out / 'gpu_measurement.json').read_text())


短测正常后，手动运行下一格开始 1000 条正式训练。5000 条是另一轮：稍后将 SIZE 改成 5000，再运行此格。两组使用不同输出目录。当前评估是 teacher-forced dev loss；任务准确率需之后用统一生成评测，不能用 loss 替代。

In [ ]:
SIZE = 1000
train_out = result_root / (f'train-{SIZE}-' + datetime.now().strftime('%Y%m%d-%H%M%S'))
subprocess.run([sys.executable, 'colab_train.py', '--size', str(SIZE), '--output', str(train_out)], check=True)
print((train_out / 'gpu_measurement.json').read_text())


如断线：重新连接、上传、安装、下载模型、挂载 Drive，跳过上述新训练格。把恢复格中的两个路径填成原输出目录和其最新完整 checkpoint 目录；参数 SIZE 必须与原运行一致。

In [ ]:
# 恢复示例：取消注释并填写真实路径后执行
# old_out = result_root / 'train-1000-实际时间'
# checkpoint = old_out / 'checkpoint-100'
# subprocess.run([sys.executable, 'colab_train.py', '--size', '1000', '--output', str(old_out), '--resume', str(checkpoint)], check=True)


训练结束后可把该次目录打包下载；Drive 中仍保留结果。之后将 adapter 和日志带回本机做同一批样本的生成评测。

In [ ]:
import shutil
last = Path(Path('colab_last_output.txt').read_text())
archive = shutil.make_archive('/content/' + last.name, 'zip', last)
files.download(archive)
